# 07/05 Tiny Trace-Specialized LSTM — 620.omnetpp_s-874B

Train/export a replay-ready **742-trainable-parameter** causal LSTM. This is a pre-run experimental design, not a claim that it already reproduces SMS. Only keyed ChampSim replay decides that.

**Policy:** region-pair ×2 + predecessor-PC ×1 + global ×1; at most four candidates/event; label lead 4–128 events; LRU dedup 256; held-out precision floor ≥0.70. SMS is strong, so this route should be judged by reproduction, coverage overlap, and resource behavior instead of raw issue count.

```mermaid
flowchart LR
 A[Current PC,line,region + past PC]-->B[region-pair/predecessor bank]
 A-->C[8 causal runtime features]
 B-->D[≤4 candidates]
 C-->E[8-unit LSTM]
 D-->F[5 candidate features]
 E-->G[concat + 8-unit projection]
 F-->G
 G-->H[utility + lead-bin heads]
 H-->I[held-out threshold/top-k]
 I-->J[LRU dedup]
 J-->K[rich list + full decision ledger]
```


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
REPO_ROOT = Path('/content/cache_arch')
REPO_URL = os.environ.get('CACHE_ARCH_REPO_URL', 'https://github.com/Angelawoo572/cache_arch.git')
if not REPO_ROOT.exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'], check=True)
print(REPO_ROOT)


In [ ]:
TRACE = '620.omnetpp_s-874B'
RUN_ID = 'tiny_trace_lstm_07_05_seed7'
SEED, MAX_ROWS, EPOCHS, CHUNK_LEN, LEDGER_SCOPE = 7, 0, 8, 1024, 'full'
ARTIFACT_DIR = REPO_ROOT / 'formal_NN_training/artifacts/tiny_trace_lstm_07_05' / TRACE / RUN_ID
oracle = REPO_ROOT / 'formal_NN_training/results/standalone_nn_data/oracle' / (TRACE + '.oracle.csv.gz')
assert oracle.is_file() and Path(str(oracle)+'.meta.json').is_file(), oracle
print(oracle, ARTIFACT_DIR)


In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'formal_NN_training/LSTM'))
from tiny_trace_prefetcher import run_trace
metadata = run_trace(repo_root=REPO_ROOT, trace=TRACE, run_id=RUN_ID, artifact_root=ARTIFACT_DIR, seed=SEED, max_rows=MAX_ROWS, epochs=EPOCHS, chunk_len=CHUNK_LEN, ledger_scope=LEDGER_SCOPE)
print(json.dumps(metadata, indent=2, sort_keys=True))
assert metadata['parameter_count'] == 742 and metadata['parameter_count'] < 1000
assert metadata['ledger_scope'] == 'full'


In [ ]:
import shutil
from google.colab import files
relative_artifact = ARTIFACT_DIR.relative_to(REPO_ROOT)
archive = shutil.make_archive('/content/' + TRACE + '_' + RUN_ID, 'zip', root_dir=str(REPO_ROOT), base_dir=str(relative_artifact))
print('rich list:', metadata['rich_list'])
print('full ledger:', metadata['decision_ledger'])
print('plan:', metadata['replay_plan'])
files.download(archive)


## After download

Unzip from the repository root on Sacramento. Use replay plus demand-event attribution to determine whether the candidate representation covers the same normal-prefetch misses as SMS and whether any difference comes from timing or resource pressure.
